# Deploy a base model

Before training or evaluating, you need a running inference endpoint. In this tutorial you'll deploy Qwen3-4B to Modal using `DeploymentConfig` and send it a generation request. Any model in the training-gym catalog works — Qwen3-4B is just a convenient default.

## Prerequisites

This tutorial requires a Modal Secret named `huggingface-secret` containing your
`HF_TOKEN`. Create one at [modal.com/secrets](https://modal.com/secrets) if you
haven't already — the cell below fails fast with instructions otherwise.

> **Note:** you do **not** need to attach a GPU to this notebook. All training and
> serving happens on Modal-managed GPU workers spun up by the SDK — the notebook
> itself only needs to issue API calls.

In [ ]:
import modal

try:
    modal.Secret.from_name("huggingface-secret").hydrate()
except modal.exception.NotFoundError as e:
    raise RuntimeError(
        "Missing Modal Secret 'huggingface-secret'. Create one at "
        "https://modal.com/secrets with an HF_TOKEN entry, then re-run."
    ) from e

In [ ]:
%uv pip install -q git+https://github.com/modal-projects/training-gym.git@main

In [ ]:
from modal_training_gym import (
    DeploymentConfig,
    Qwen3_4B,
)

## Serve the base model

`DeploymentConfig.serve()` launches an SGLang inference server on Modal and returns a deployment handle you can call directly.

In [ ]:
base_model = Qwen3_4B()
base_model_deployment = DeploymentConfig(
    model=base_model,
).serve()
print(f"Base model deployed to {base_model_deployment.url}")

The first call cold-starts the SGLang server, so expect it to take a few minutes. Subsequent requests are fast.

In [ ]:
response = base_model_deployment.generate(
    "Write a short story about a cat.",
)
print(response)

## Disabling thinking mode

Qwen3 defaults to "thinking" mode, where the model reasons internally before answering. Pass `enable_thinking=False` via `chat_template_kwargs` to get a direct response instead.

In [ ]:
response = base_model_deployment.generate(
    "Write a short story about a cat.",
    chat_template_kwargs={"enable_thinking": False},
)
print(response)

## Check the dashboard

Your deployment should now be visible in the training-gym dashboard you set up earlier.

In [ ]:
print(base_model_deployment.modal_app_url)